In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
    IntegerType
)
from delta.tables import DeltaTable

In [0]:
catalog = "sales_catalog"
schema = "main"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
# ── 1. Insert Users ────────────────────────
users_data = [
    ("U001", "alice",   "alice@gmail.com",   1500.00),
    ("U002", "bob",     "bob@gmail.com",      800.00),
    ("U003", "charlie", "charlie@gmail.com",  300.00),
    ("U004", "diana",   "diana@gmail.com",   2000.00),
    ("U005", "eve",     "eve@gmail.com",      150.00),
]

users_schema = StructType([
    StructField("user_id",  StringType(), nullable=False),
    StructField("username", StringType(), nullable=False),
    StructField("email",    StringType(), nullable=False),
    StructField("balance",  DoubleType(), nullable=False)
])

users_df = spark.createDataFrame(users_data, users_schema)
DeltaTable.forName(spark, "users").alias("target") \
    .merge(users_df.alias("source"), "target.user_id = source.user_id") \
    .whenNotMatchedInsertAll() \
    .execute()


In [0]:
# ── 2. Insert Items ────────────────────────
items_data = [
    ("I001", "laptop",     999.99),
    ("I002", "phone",      499.99),
    ("I003", "tablet",     299.99),
    ("I004", "headphones",  79.99),
    ("I005", "charger",     29.99),
]

items_schema = StructType([
    StructField("item_id",   StringType(), nullable=False),
    StructField("item_name", StringType(), nullable=False),
    StructField("price",     DoubleType(), nullable=False)
])

items_df = spark.createDataFrame(items_data, items_schema)

DeltaTable.forName(spark, "items").alias("target") \
    .merge(items_df.alias("source"), "target.item_id = source.item_id") \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
# ── 3. Insert Inventory ────────────────────
inventory_data = [
    ("I001", 10),
    ("I002", 25),
    ("I003", 15),
    ("I004", 50),
    ("I005", 100),
]

inventory_schema = StructType([
    StructField("item_id",            StringType(),  nullable=False),
    StructField("quantity_available", IntegerType(), nullable=False)
])

inventory_df = spark.createDataFrame(inventory_data, inventory_schema)

DeltaTable.forName(spark, "inventory").alias("target") \
    .merge(inventory_df.alias("source"), "target.item_id = source.item_id") \
    .whenNotMatchedInsertAll() \
    .execute()